<a href="https://colab.research.google.com/github/Murcha1990/ML_AI26/blob/main/Week4_Classification/ClassificationAndEncoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  План семинара

1. Линейный классификатор в задаче бинарной классификации
2. Кодирование категориальных признаков
3. Метрики качества: матрица ошибок, precision, recall, F1, F_β и выбор порога под бизнес-задачу
4. Подбор гиперпараметров логистической регрессии (GridSearchCV)

## Задача бинарной классификации

In [ ]:
# seaborn уже установлен в Colab; если нет, раскомментируйте
# !pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
np.random.seed(42)

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/Murcha1990/ML_AI24/refs/heads/main/Lesson9_ClassificationBase/bike_buyers_clean.csv')

In [ ]:
data

# Обзор данных

In [ ]:
# проверим типы колонок в датасете
data.dtypes

In [ ]:
X = data.iloc[:, :-1].drop(columns='ID')

y = data['Purchased Bike']

In [ ]:
# select_dtypes работает и в pandas 2 (строки = object), и в pandas 3 (строки = str)
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns

print(f"We have {len(num_cols)} numeric columns: {', '.join(num_cols)}")
print(f"And {len(cat_cols)} categorical columns: {', '.join(cat_cols)}")

In [ ]:
for col in cat_cols:
    print(col)
    display(X[col].value_counts(normalize=True))
    print()

In [ ]:
# у нас есть категориальные переменные разных видов!

binary_cols = cat_cols[X[cat_cols].nunique() == 2].tolist()
ordinal_cols = ['Commute Distance', 'Education']
cat_cols = cat_cols.difference(binary_cols + ordinal_cols).tolist()

In [ ]:
X.describe()

In [ ]:
X.describe(exclude='number')

In [ ]:
# classes are balanced !
y.value_counts(normalize=True)

In [ ]:
y

In [ ]:
# transform y to numeric column
y = (y == 'Yes').astype(int)
y

# Подготовка данных

## Кодирование категориальных признаков

In [ ]:
# в Colab category_encoders не установлен по умолчанию
!pip install -q category_encoders

In [ ]:
from category_encoders.ordinal import OrdinalEncoder # LabelEncoder
from category_encoders.one_hot import OneHotEncoder # OneHotEncoding
from category_encoders.target_encoder import TargetEncoder # счетчики+сглаживание

In [ ]:
X['Education'].unique()

In [ ]:
# Ordinal: from categories to numbers

ord_enc = OrdinalEncoder()
ord_enc.fit_transform(X['Education'])

In [ ]:
# One hot: from k categories to k dummy columns

one_hot_enc = OneHotEncoder()

one_hot_enc.fit_transform(X['Education'])
# * fit -> определить количество новых столбцов (по кол-ву категорий)
# * transform -> создать новые столбцы
# * fit_transform = fit + transform

# Нужно ли удалять какую-то из колонок после такого кодирования ?

Target encoding вычисляет значения по формуле

$$\frac{mean(target)\cdot n_{rows} + \alpha \cdot globalMean}{n_{rows} + \alpha} $$

In [ ]:
# target encoding: from k categories to posterior probabilites of y == 1 - P(y==1 | category == c1)

tgt_enc = TargetEncoder(smoothing=1)

# smoothing - это коэффициент сглаживания alpha, чем он больше, тем больше регуляризация

tgt_enc.fit_transform(X['Education'], y)

In [ ]:
# энкодер можно применять сразу на весь датафрейм

tgt_enc = TargetEncoder(cols=['Education', 'Gender', 'Region'])
tgt_enc.fit_transform(X, y)

Помимо сглаживания, для борьбы с переобучением при таргет энкодинге в лекции предлагались и другие методы

- Добавление случайного шума
- Вычисление счетчиков на кросс-валидации
- Expanding mean encoding

Первые две идеи реализованы в классе LeaveOneOut

- значения считаются на основе кросс-валидации вида leave one out (то есть значение энкодинга для конкретного наблюдения будет считаться по всем наблюдениям, кроме этого)
- параметр sigma отвечает за дисперсию случайного шума, который добавляется к значению энкодинга (чем больше sigma, тем больше регуляризация)

In [ ]:
from category_encoders.leave_one_out import LeaveOneOutEncoder

loo_enc = LeaveOneOutEncoder(sigma=3.)

loo_enc.fit_transform(X['Education'], y)

## Масштабирование числовых признаков

In [ ]:
X['Income']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler

scaler = StandardScaler()  # x -> (x - mean) / std
scaler.fit_transform(X[['Income']])[:5]

# fit -> вычисляет параметры преобразования: mean, std (только по TRAIN)
# transform -> преобразует столбец по формуле (и TRAIN, и TEST)
# scaler.fit(X_train); scaler.transform(X_train); scaler.transform(X_test)

Параметры, которые `StandardScaler` запомнил на этапе `fit`:

In [ ]:
print('mean:', scaler.mean_)
print('var: ', scaler.var_)

Другие популярные варианты масштабирования:

* `MinMaxScaler`: $x \to \frac{x - \min}{\max - \min}$, значения попадают в отрезок $[0, 1]$;
* `MaxAbsScaler`: $x \to \frac{x}{\max |x|}$, значения попадают в $[-1, 1]$, нули остаются нулями (удобно для разреженных данных).

In [ ]:
pd.DataFrame({
    'Income': X['Income'].values[:5],
    'StandardScaler': StandardScaler().fit_transform(X[['Income']])[:5, 0],
    'MinMaxScaler': MinMaxScaler().fit_transform(X[['Income']])[:5, 0],
    'MaxAbsScaler': MaxAbsScaler().fit_transform(X[['Income']])[:5, 0],
}).round(3)

Есть две проблемы:
- класc StandardScaler не умеет работать только на части колонок датафрейма
- классы sklearn возвращают numpy arrays, а не pandas dataframe, что не удобно

In [ ]:
num_cols

In [ ]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer([('scaler', StandardScaler(), num_cols)], remainder='passthrough') # 'drop'

In [ ]:
ct.fit_transform(X)

In [ ]:
# нет удобной реализации - напишем сами !

from sklearn.base import TransformerMixin

class CustomScaler(TransformerMixin):
    def __init__(self, cols, scaler=None):
        self.cols = cols
        self.scaler = scaler or StandardScaler()

    def fit(self, X, y=None):
        num_cols = X.copy()[self.cols]
        self.scaler.fit(num_cols)
        return self
    def transform(self, X, y=None):
        X_res = X.copy()
        num_cols_tr = self.scaler.transform(X_res[self.cols])
        for i, col in enumerate(self.cols):
            X_res[col] = num_cols_tr[:,i]
        return X_res

In [ ]:
sc = CustomScaler(num_cols)
X2 = sc.fit_transform(X)

In [ ]:
X2

# Соберем все преобразования данных в pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

p1 = Pipeline([
    ('ordinal_encoder_', OrdinalEncoder(cols=ordinal_cols + binary_cols + cat_cols)), # плохо!!!
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
    ])

p2 = Pipeline([
    ('one_hot_encoder_', OneHotEncoder(cols=ordinal_cols + binary_cols+cat_cols)),
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
    ])

p3 = Pipeline([
    ('target_encoder_', TargetEncoder(cols=ordinal_cols + binary_cols+cat_cols)),
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
])

p4 = Pipeline([
    ('ordinal_encoder_', OrdinalEncoder(cols=ordinal_cols)),
    ('one_hot_encoder_', OneHotEncoder(cols=binary_cols+cat_cols)),
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
    ])

p5 = Pipeline([
    ('ordinal_encoder_', OrdinalEncoder(cols=ordinal_cols)),
    ('one_hot_encoder_', OneHotEncoder(cols=binary_cols)),
    ('target_encoder_', TargetEncoder(cols=cat_cols)),
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
])

p6 = Pipeline([
    ('one_hot_encoder_', OneHotEncoder(cols=binary_cols)),
    ('target_encoder_', TargetEncoder(cols=cat_cols + ordinal_cols)),
    ('scaler_', CustomScaler(num_cols)),
    ('model_', LogisticRegression())
])

In [ ]:
# пример работы с пайплайном
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

p1.fit(X_train, y_train)

#print(p1)

y_pred = p1.predict(X_test)

print(accuracy_score(y_test, y_pred))

# Сравнение качества классификации при разных пайплайнах преобразования данных

Вообще существует довольно большое количество метрик для задачи бинарной классификации.

Но для нашей задачи разберем самую простую и интуитивную метрику: accuracy

$accuracy = \frac{1}{n}\Sigma_{i=0}^n [\hat y_i == y_i]$

То есть доля правильных предсказаний

In [ ]:
from sklearn.model_selection import cross_validate, cross_val_score
import warnings

warnings.filterwarnings('ignore')

In [ ]:
for i, pipe in enumerate([p1, p2, p3, p4, p5, p6]):
    cv_res = cross_validate(pipe,
                            X,
                            y,
                            cv=5,
                            scoring='accuracy'
                           )
    print(f"Pipeline {i + 1}: mean cv accuracy = {cv_res['test_score'].mean()}")

In [ ]:
p3.fit(X_train, y_train)

pred = p3.predict(X_test)

accuracy_score(y_test, pred)

# Метрики качества классификации

Accuracy показывает долю правильных ответов, но ничего не говорит о том, **какие именно** ошибки делает модель. А в реальной задаче ошибки разного типа обычно стоят по-разному. Поэтому сначала договоримся, зачем бизнесу эта модель.

## Бизнес-постановка

Сеть веломагазинов готовит рассылку: клиентам из базы отправляют персональное предложение — скидку на велосипед и приглашение на бесплатный тест-драйв. Модель решает, **кому отправить предложение**: тем, у кого предсказан класс 1 («купит велосипед»).

Модель ошибается двумя способами:

* **Ложное срабатывание (False Positive, FP):** предложение получил человек, который велосипед не купит. Мы зря потратили деньги на контакт: печать и доставку каталога, звонок менеджера, организацию тест-драйва. Пусть это **1000 ₽** на человека.
* **Пропуск (False Negative, FN):** мы не отправили предложение человеку, который купил бы велосипед. Он уходит к конкурентам, и мы теряем маржу с продажи. Пусть это **2000 ₽**.

Пропустить покупателя в 2 раза дороже, чем зря его побеспокоить. Значит, нам важнее найти как можно больше будущих покупателей, то есть **recall важнее precision**.

Совсем забыть про precision нельзя: если отправить предложение всем клиентам, recall будет равен 1, но бюджет рассылки уйдёт в основном на тех, кто ничего не купит.

Цифры, конечно, условные: в реальной задаче их берут у бизнеса (стоимость контакта, средняя маржа, доля откликнувшихся на предложение).

In [ ]:
COST_FP = 1000   # лишний контакт: человек получил предложение, но не купит
COST_FN = 2000   # пропущенный покупатель: упущенная маржа

def business_cost(y_true, y_pred):
    # суммарные потери на выборке в рублях
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * COST_FP + fn * COST_FN

## Матрица ошибок

Матрица ошибок (confusion matrix) раскладывает все предсказания на четыре группы. В sklearn строки — это **правильные ответы**, столбцы — **предсказания**:

|  | предсказали 0 (не отправляем) | предсказали 1 (отправляем) |
|---|---|---|
| **на самом деле 0** (не купит) | TN — верно не побеспокоили | FP — лишний контакт, −1000 ₽ |
| **на самом деле 1** (купит) | FN — пропустили покупателя, −2000 ₽ | TP — нашли покупателя |

Построим матрицу ошибок для пайплайна `p3` с обычным порогом 0.5 (его мы обучили выше на `X_train`).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred_05 = p3.predict(X_test)   # predict = порог 0.5 по вероятности класса 1

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_05,
    display_labels=['не купит (0)', 'купит (1)'],
    cmap='Blues', colorbar=False
)
plt.xlabel('Предсказание модели')
plt.ylabel('Правильный ответ')
plt.title('Матрица ошибок, порог 0.5')
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, pred_05).ravel()
print(f'TP = {tp}: нашли будущих покупателей')
print(f'FP = {fp}: лишние контакты, {fp * COST_FP} ₽')
print(f'FN = {fn}: пропущенные покупатели, {fn * COST_FN} ₽')
print(f'TN = {tn}: верно не стали беспокоить')
print(f'Потери на тесте: {business_cost(y_test, pred_05)} ₽')

## Precision, Recall, F1, F_β

Все эти метрики считаются по матрице ошибок.

**Precision (точность)** — какая доля тех, кому мы отправили предложение, действительно купит:

$$Precision = \frac{TP}{TP + FP}$$

В нашей задаче это «эффективность рассылки»: сколько контактов потрачено не зря.

**Recall (полнота)** — какую долю всех будущих покупателей мы нашли:

$$Recall = \frac{TP}{TP + FN}$$

В нашей задаче это «охват»: сколько потенциальных покупателей получили предложение.

**F1-мера** — среднее гармоническое precision и recall. Она высокая, только когда обе метрики высокие, и считает их **одинаково важными**:

$$F_1 = \frac{2 \cdot Precision \cdot Recall}{Precision + Recall}$$

**F_β-мера** обобщает F1 и позволяет сказать, во сколько раз recall важнее precision:

$$F_\beta = \frac{(1+\beta^2) \cdot Precision \cdot Recall}{\beta^2 \cdot Precision + Recall} = \frac{(1+\beta^2)\,TP}{(1+\beta^2)\,TP + \beta^2\,FN + FP}$$

* $\beta = 1$ — обычная F1;
* $\beta > 1$ — важнее recall (например, $F_2$);
* $\beta < 1$ — важнее precision (например, $F_{0.5}$).

### Какое β выбрать

Посмотрите на правую часть формулы: пропуски FN входят в знаменатель с весом $\beta^2$, а ложные срабатывания FP — с весом 1. Поэтому естественно взять

$$\beta^2 = \frac{\text{цена FN}}{\text{цена FP}} = \frac{2000}{1000} = 2 \quad\Rightarrow\quad \beta = \sqrt{2} \approx 1.41.$$

Итак, метрика, которая отражает приоритеты нашей задачи, — **$F_\beta$ с $\beta \approx 1.41$**. β не обязано быть целым: $F_2$ и $F_{0.5}$ — просто самые популярные варианты. Если бы пропуск стоил в 4 раза дороже лишнего контакта, мы взяли бы $F_2$.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, accuracy_score

BETA = np.sqrt(COST_FN / COST_FP)
print('beta =', BETA)

# считаем вручную по матрице ошибок...
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
fb = (1 + BETA**2) * precision * recall / (BETA**2 * precision + recall)

# ...и сверяем со sklearn
pd.DataFrame({
    'вручную': [precision, recall, f1, fb],
    'sklearn': [precision_score(y_test, pred_05), recall_score(y_test, pred_05),
                f1_score(y_test, pred_05), fbeta_score(y_test, pred_05, beta=BETA)],
}, index=['precision', 'recall', 'F1', 'F_beta']).round(4)

Все метрики сразу (по каждому классу) печатает `classification_report`. Строка класса 1 — это precision и recall, которые мы считали выше. Строка класса 0 — те же метрики, если считать «положительным» класс 0.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_05, target_names=['не купит (0)', 'купит (1)']))

## Как метрики зависят от порога

`predict` относит объект к классу 1, если предсказанная вероятность больше 0.5. Но 0.5 — не закон природы. Если **понизить порог**, предложение получит больше людей: recall вырастет, precision упадёт. Если повысить — наоборот.

**Важно: порог — это тоже параметр модели, и подбирать его на тестовой выборке нельзя.** Иначе оценка качества на тесте получится слишком оптимистичной. Подберём порог на обучающей выборке с помощью кросс-валидации: `cross_val_predict` возвращает для каждого объекта `X_train` вероятность, предсказанную моделью, которая этот объект **не видела** (out-of-fold предсказания). Тест оставим для финальной проверки.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(p3, X_train, y_train, cv=cv5, method='predict_proba')[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
rows = []
for thr in thresholds:
    pr = (oof_probs >= thr).astype(int)
    rows.append({
        'threshold': thr,
        'precision': precision_score(y_train, pr, zero_division=0),
        'recall': recall_score(y_train, pr),
        'F1': f1_score(y_train, pr),
        'F_beta': fbeta_score(y_train, pr, beta=BETA),
        'cost': business_cost(y_train, pr),
    })
thr_df = pd.DataFrame(rows)

best_thr = thr_df.loc[thr_df['F_beta'].idxmax(), 'threshold']
best_cost_thr = thr_df.loc[thr_df['cost'].idxmin(), 'threshold']
print(f'Порог с максимальной F_beta: {best_thr:.2f}')
print(f'Порог с минимальными потерями: {best_cost_thr:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for col in ['precision', 'recall', 'F1', 'F_beta']:
    axes[0].plot(thr_df['threshold'], thr_df[col], label=col, lw=2.5 if col == 'F_beta' else 1.5)
axes[0].axvline(0.5, c='gray', ls='--', label='порог 0.5')
axes[0].axvline(best_thr, c='red', ls='--', label=f'макс. F_beta: {best_thr:.2f}')
axes[0].set_xlabel('порог')
axes[0].set_title('Метрики на кросс-валидации (train)')
axes[0].legend()

axes[1].plot(thr_df['threshold'], thr_df['cost'] / 1000)
axes[1].axvline(0.5, c='gray', ls='--', label='порог 0.5')
axes[1].axvline(best_thr, c='red', ls='--', label=f'макс. F_beta: {best_thr:.2f}')
axes[1].axvline(best_cost_thr, c='green', ls=':', label=f'мин. потери: {best_cost_thr:.2f}')
axes[1].set_xlabel('порог')
axes[1].set_ylabel('потери, тыс. ₽')
axes[1].set_title('Потери бизнеса на кросс-валидации (train)')
axes[1].legend()
plt.show()

Что видно на графиках:

* при уменьшении порога recall растёт, а precision падает — это и есть компромисс между ними;
* максимум F1 и максимум $F_\beta$ находятся в разных местах: $F_\beta$ «тянет» порог вниз, потому что пропуски для неё важнее;
* **порог с максимальной $F_\beta$ и порог с минимальными потерями могут сильно отличаться.** Правило $\beta^2 = \frac{\text{цена FN}}{\text{цена FP}}$ правильно задаёт, что нам важнее, но $F_\beta$ — это не деньги.

Почему так. Посмотрим на стратегию «отправить всем»: recall = 1, а precision равна доле покупателей $\pi$ (у нас $\pi \approx 0.48$). Тогда

$$F_\beta(\text{всем}) = \frac{(1+\beta^2)\,\pi}{\beta^2\pi + 1} \approx \frac{3 \cdot 0.48}{2 \cdot 0.48 + 1} \approx 0.73.$$

Это очень высокое значение. Модели с accuracy около 0.65 побить его очень трудно: когда классы сбалансированы, а модель не очень сильная, $F_\beta$ с $\beta \ge 1$ почти всегда выше у тривиальной стратегии «всем — класс 1» (даже F1 при $\pi = 0.48$ у неё около 0.65). Деньги же честно считают каждый лишний контакт.

Поэтому разделим роли:
* **$F_\beta$** показывает, насколько модель соответствует приоритетам бизнеса, и её удобно использовать для сравнения порогов и моделей **с обязательной проверкой против тривиальной стратегии**;
* **итоговый порог** выбираем по прямой бизнес-метрике — потерям в рублях, если её можно посчитать.

Для порога по потерям есть и теоретическая оценка. Если модель выдаёт честные вероятности, отправлять предложение выгодно, когда ожидаемая потеря от пропуска больше ожидаемой потери от лишнего контакта: $p \cdot 2000 > (1-p) \cdot 1000$, то есть $p > \frac{1000}{1000 + 2000} = \frac{1}{3}$. В общем виде оптимальный порог равен $\frac{\text{цена FP}}{\text{цена FP} + \text{цена FN}}$. Сравните его с порогом минимальных потерь на графике.

Тот же компромисс удобно смотреть на **PR-кривой** (precision–recall curve): каждая её точка — пара (recall, precision) при каком-то пороге.

In [ ]:
from sklearn.metrics import precision_recall_curve

prec_curve, rec_curve, _ = precision_recall_curve(y_train, oof_probs)

plt.figure(figsize=(8, 6))
plt.plot(rec_curve, prec_curve, label='PR-кривая')
for thr, color, name in [(0.5, 'gray', 'порог 0.5'),
                         (best_thr, 'red', f'макс. F_beta, порог {best_thr:.2f}'),
                         (best_cost_thr, 'green', f'мин. потери, порог {best_cost_thr:.2f}')]:
    pr = (oof_probs >= thr).astype(int)
    plt.scatter(recall_score(y_train, pr), precision_score(y_train, pr), s=120, c=color, zorder=3, label=name)
plt.axhline(y_train.mean(), c='black', lw=0.8, ls=':', label='случайная модель (доля класса 1)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR-кривая на кросс-валидации (train)')
plt.legend()
plt.show()

Теперь проверим выбранные пороги на **тестовой** выборке, которую мы при подборе не использовали. Для сравнения добавим тривиальную стратегию «отправить всем».

In [ ]:
def report(y_true, probs, thr):
    pr = (probs >= thr).astype(int)
    return {
        'accuracy': accuracy_score(y_true, pr),
        'precision': precision_score(y_true, pr, zero_division=0),
        'recall': recall_score(y_true, pr),
        'F1': f1_score(y_true, pr),
        'F_beta': fbeta_score(y_true, pr, beta=BETA),
        'потери, ₽': business_cost(y_true, pr),
    }

test_probs = p3.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, thr in zip(axes, [0.5, best_thr, best_cost_thr]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, (test_probs >= thr).astype(int),
        display_labels=['не купит (0)', 'купит (1)'], cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(f'Тест, порог {thr:.2f}')
plt.tight_layout()
plt.show()

pd.DataFrame({
    'отправить всем': report(y_test, test_probs, 0.0),
    'порог 0.5': report(y_test, test_probs, 0.5),
    f'порог {best_thr:.2f} (макс. F_beta)': report(y_test, test_probs, best_thr),
    f'порог {best_cost_thr:.2f} (мин. потери)': report(y_test, test_probs, best_cost_thr),
}).round(3)

Как читать таблицу:

* при пониженном пороге модель находит больше будущих покупателей (recall растёт) ценой большего числа лишних контактов (precision падает). Accuracy при этом может даже снизиться, но для бизнеса важна не она, а потери. Это хорошая иллюстрация того, почему accuracy — не лучшая метрика, когда ошибки стоят по-разному;
* «отправить всем» — стратегия без модели. По $F_\beta$ она легко может оказаться лучшей, но по деньгам модель полезна, только если её потери меньше. Если разница небольшая, модели не хватает качества, и её стоит улучшать: признаки, гиперпараметры, другая модель.

> В sklearn 1.5+ подбор порога по кросс-валидации реализован в классе `TunedThresholdClassifierCV`. Например, `TunedThresholdClassifierCV(p3, scoring=make_scorer(business_cost, greater_is_better=False))` делает то же, что мы сделали руками для потерь.

# Подбор гиперпараметров логистической регрессии

До сих пор логистическая регрессия работала с параметрами по умолчанию. Подберём основные гиперпараметры с помощью `GridSearchCV`.

Что будем перебирать:

* **`C`** — обратная сила регуляризации: чем меньше `C`, тем сильнее штраф за большие веса и тем проще модель. Перебираем по логарифмической сетке.
* **Тип регуляризации** — L2 (веса маленькие, но ненулевые), L1 (часть весов зануляется, то есть происходит отбор признаков) или их смесь elastic-net. Задаётся параметром `l1_ratio`: `0` — L2, `1` — L1, между ними — elastic-net.
* **`class_weight`** — веса классов. При `'balanced'` ошибки на каждом классе взвешиваются обратно пропорционально его частоте. У нас классы почти сбалансированы, так что заметного эффекта ждать не стоит, но при дисбалансе это один из главных параметров.
* **`solver='saga'`** — единственный солвер, который поддерживает все три вида регуляризации. Он стохастический и требует признаков одного масштаба: числовые признаки у нас стандартизованы, а target encoding даёт значения от 0 до 1.

**По какой метрике подбирать.** Подбирать гиперпараметры по $F_\beta$ при пороге 0.5 опасно: как мы видели, её легко «обмануть», предсказывая всем класс 1. При очень сильной регуляризации (маленьком `C`) модель именно так и делает, и `GridSearchCV` может выбрать такую бесполезную модель. Поэтому гиперпараметры выберем по **log loss** — функции потерь, которую минимизирует сама логистическая регрессия. Она оценивает качество предсказанных вероятностей и не зависит от порога. Порог потом подберём под бизнес, как раньше. $F_\beta$ и потери тоже посчитаем на кросс-валидации, чтобы видеть их рядом.

В sklearn метрики в `GridSearchCV` всегда максимизируются, поэтому log loss передаётся со знаком минус: `'neg_log_loss'`.

Перебор делаем на `X_train` по кросс-валидации; target encoding внутри пайплайна заново обучается на каждом фолде, поэтому утечки целевой переменной нет. Тест по-прежнему не трогаем.

> Начиная со sklearn 1.8 параметр `penalty` объявлен устаревшим: вид регуляризации задаётся одним `l1_ratio`. В более старых версиях для `l1_ratio` нужно указать `penalty='elasticnet'` — так мы и сделаем, чтобы код работал в обеих версиях.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

scoring = {
    'log_loss': 'neg_log_loss',                                     # log loss со знаком минус
    'f_beta': make_scorer(fbeta_score, beta=BETA),
    'cost': make_scorer(business_cost, greater_is_better=False),   # sklearn максимизирует, поэтому потери со знаком минус
}

pipe_lr = Pipeline([
    ('target_encoder', TargetEncoder(cols=ordinal_cols + binary_cols + cat_cols)),
    ('scaler', CustomScaler(num_cols)),
    ('model', LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.0, max_iter=5000))
])

param_grid = {
    'model__C': np.logspace(-3, 2, 6),            # 0.001, 0.01, ..., 100
    'model__l1_ratio': [0.0, 0.5, 1.0],           # L2, elastic-net, L1
    'model__class_weight': [None, 'balanced'],
}
# имя параметра = имя шага в пайплайне ('model') + '__' + имя параметра модели.
# Поэтому шаги здесь названы без подчёркивания в конце: 'model_' + '__C' дало бы 'model___C', и sklearn не разобрал бы имя.

grid = GridSearchCV(pipe_lr, param_grid, scoring=scoring, refit='log_loss', cv=cv5, n_jobs=-1)
grid.fit(X_train, y_train)

print('Лучшие параметры по log loss:', grid.best_params_)
print(f'Log loss на кросс-валидации: {-grid.best_score_:.4f}')

res = pd.DataFrame(grid.cv_results_)
res['потери на фолде, ₽'] = -res['mean_test_cost']
res['log loss'] = -res['mean_test_log_loss']
res['param_model__class_weight'] = res['param_model__class_weight'].map(lambda v: 'balanced' if v == 'balanced' else 'None')
cols_show = ['param_model__C', 'param_model__l1_ratio', 'param_model__class_weight',
             'log loss', 'std_test_log_loss', 'mean_test_f_beta', 'потери на фолде, ₽']
res[cols_show].sort_values('log loss').head(10).round(4)

Посмотрим на результаты перебора целиком: как log loss и $F_\beta$ (при пороге 0.5) зависят от `C` и типа регуляризации.

In [ ]:
import seaborn as sns

res['class_weight'] = res['param_model__class_weight']

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for row, (metric, title, cmap) in enumerate([('log loss', 'log loss (меньше — лучше)', 'viridis_r'),
                                             ('mean_test_f_beta', 'F_beta при пороге 0.5', 'viridis')]):
    for col, cw in enumerate(['None', 'balanced']):
        ax = axes[row, col]
        table = res[res['class_weight'] == cw].pivot_table(
            index='param_model__l1_ratio', columns='param_model__C', values=metric)
        sns.heatmap(table, annot=True, fmt='.3f', cmap=cmap, ax=ax,
                    vmin=res[metric].min(), vmax=res[metric].max())
        ax.set_title(f'{title}, class_weight = {cw}')
        ax.set_xlabel('C')
        ax.set_ylabel('l1_ratio (0 = L2, 1 = L1)')
        ax.set_xticklabels([f'{c:g}' for c in table.columns])
plt.tight_layout()
plt.show()

Как читать эти картинки:

* при очень маленьком `C` регуляризация настолько сильная, что веса почти нулевые, и модель предсказывает почти константу. Особенно быстро это происходит с L1: она зануляет веса целиком. Log loss в таких клетках растёт, а $F_\beta$ может, наоборот, оказаться высокой — если константа получилась «класс 1 для всех». Это та самая ловушка, из-за которой мы не подбираем гиперпараметры по $F_\beta$ при пороге 0.5;
* при больших `C` регуляризация почти не действует, и все варианты дают похожий результат;
* разница между хорошими вариантами часто сравнима с разбросом по фолдам (`std_test_log_loss`), поэтому делать выводы по третьему знаку не стоит.

Для лучшей модели подберём порог по потерям — так же, как раньше, по кросс-валидации на train — и сравним на тесте все варианты.

In [ ]:
best_lr = grid.best_estimator_

oof_best = cross_val_predict(best_lr, X_train, y_train, cv=cv5, method='predict_proba')[:, 1]
cost_by_thr = [business_cost(y_train, (oof_best >= t).astype(int)) for t in thresholds]
best_cost_thr_lr = thresholds[int(np.argmin(cost_by_thr))]
print(f'Лучшая модель: порог с минимальными потерями {best_cost_thr_lr:.2f}')

test_probs_best = best_lr.predict_proba(X_test)[:, 1]

pd.DataFrame({
    'отправить всем': report(y_test, test_probs, 0.0),
    'p3, порог 0.5': report(y_test, test_probs, 0.5),
    f'p3, порог {best_cost_thr:.2f}': report(y_test, test_probs, best_cost_thr),
    'GridSearch, порог 0.5': report(y_test, test_probs_best, 0.5),
    f'GridSearch, порог {best_cost_thr_lr:.2f}': report(y_test, test_probs_best, best_cost_thr_lr),
}).round(3)

## Выводы

* Матрица ошибок показывает, **какие** ошибки делает модель, а не только сколько их. Accuracy этого не видит.
* Метрику нужно выбирать под задачу. Если пропуск дороже ложного срабатывания, важнее recall, и для сравнения удобно использовать $F_\beta$ с $\beta^2 = \frac{\text{цена FN}}{\text{цена FP}}$.
* $F_\beta$ — не деньги. При сбалансированных классах и не очень сильной модели её максимизирует тривиальная стратегия «всем — класс 1». Всегда сравнивайте модель с такой стратегией, а если цену ошибок можно посчитать, выбирайте порог по прямой бизнес-метрике.
* Порог классификации — такой же параметр модели, как `C`. Его подбирают по кросс-валидации на обучающей выборке, а не на тесте.
* Гиперпараметры логистической регрессии (`C`, тип регуляризации, `class_weight`) удобно подбирать по метрике, не зависящей от порога, например по log loss, а порог настраивать отдельно под бизнес.